# **ECGAssistant**

# Definición del Problema

## ¿Qué problema resolvemos?

Las enfermedades cardiovasculares son la principal causa de muerte a nivel mundial.
Un electrocardiograma (ECG) es la herramienta más usada para detectar arritmias,
pero su interpretación requiere un cardiólogo especializado, lo que limita el acceso
en contextos de escasez de personal médico o atención primaria.

El objetivo de este proyecto es construir un sistema de apoyo al diagnóstico que:

1. Clasifique automáticamente latidos cardíacos como normales o anormales a partir
   de señales ECG crudas.
2. Genere explicaciones clínicas en español sobre el resultado, ancladas en
   documentación médica real.
3. No sustituya al médico, sino que actúe como un primer filtro que señale qué
   pacientes requieren atención urgente.




## Pipeline técnico

El sistema combina tres tecnologías:

- **Deep Learning (CNN 1D):** en lugar de calcular manualmente características de la señal (altura del pico R, duración del QRS, etc.), la red aprende sola qué patrones son relevantes para distinguir un latido normal de uno anormal directamente a partir de los voltajes crudos del ECG.
- **Fine-tuning con QLoRA (Gemma 3 4B):** genera explicaciones clínicas en español
  sin alucinar información médica.
- **RAG con FAISS:** recupera contexto de guías clínicas actualizadas para enriquecer
  las explicaciones sin reentrenar el modelo.

## Métricas de éxito


La métrica principal que usamos para evaluar el modelo es el F1-score sobre la clase anormal, no la accuracy. El motivo es por que nuestro dataset tiene un desbalance de clases (~70% latidos normales, ~30% anormales). Un modelo que predijera siempre "normal" sin aprender nada obtendría un 70% de accuracy, lo que parece un buen resultado pero sería completamente inútil, ya que habría fallado en todos los pacientes con arritmia.

El F1-score evita este problema porque para obtener una puntuación alta el modelo tiene que cumplir dos condiciones a la vez: detectar la mayoría de las arritmias reales (recall alto) y no generar demasiadas falsas alarmas (precisión alta). No puede "hacer trampa" prediciendo siempre la clase más frecuente.

En un contexto médico esto es especialmente importante, ya que un falso negativo (una arritmia que el sistema no detecta) tiene consecuencias clínicas mucho más graves que una falsa alarma.
Objetivo marcado: F1 ≥ 0.95 sobre el conjunto de test.

## Limitaciones y consideraciones éticas

- El sistema está diseñado como herramienta de **apoyo**, no de diagnóstico definitivo.
- Entrenado exclusivamente sobre MIT-BIH (2 derivaciones). El rendimiento puede variar
  en ECGs de 12 derivaciones o dispositivos wearables.
- Los datos del dataset son reales pero anonimizados, cumpliendo con los principios
  de privacidad médica.
- Toda respuesta del sistema incluye un aviso explícito de que no reemplaza la
  evaluación de un profesional médico.